# Testing Notebook

## Setup
* Grab the dataset, import required packages, and check if we are running on GPU

In [1]:
!pip install transformers accelerate sentencepiece datasets==2.19.0 -q

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import math
from tqdm import tqdm

In [3]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: False
Device: CPU


## Checking for Hugging Face:

In [5]:
from huggingface_hub import login
login(<API_KEY>) # API key not shown for privacy

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /tmp/xdg-cache/huggingface/token
Login successful


## Intialize our Model and AutoTokenizer

In [ ]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
# model.config.pad_token_id = model.config.eos_token_id


model.eval()
print("Model loaded.")

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading model...


...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### Using IA3 (comment out when not being used)

In [ ]:
from peft import IA3Config, get_peft_model

# Apply IA3
peft_config = IA3Config()
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() 

## Loading the entire MedQA dataset of multiple choice questions

In [ ]:
print("Loading MedQA...")

# Smaller version so your GPU stays alive
ds = load_dataset("bigbio/med_qa", "med_qa_en_4options_bigbio_qa")["train"][:200]

### Putting Questions in Cleaner Format 

In [ ]:
processed = []

for q, opts, ans_text in zip(ds["question"], ds["choices"], ds["answer"]):
    
    # ans_text is a list like ["Nitrofurantoin"]
    correct_text = ans_text[0]

    # find the index of the correct answer in choices
    try:
        correct_idx = opts.index(correct_text)
    except ValueError:
        # skip weird data
        continue

    # convert index → letter
    correct_letter = chr(ord("A") + correct_idx)
    
    processed.append({
        "question": q,
        "options": opts,
        "answer": correct_letter
    })

## Functions for ECE, Format Query (for CoT or System Prompting), & Score Option (for selecting answer)

In [ ]:
def expected_calibration_error(confidences, correctness, num_bins=15):
    """
    Calculates the Expected Calibration Error (ECE) of a model's predictions.
    Args:
        confidences (list of float): The predicted confidence scores for each prediction.
        correctness (list of int): Binary indicators (1 or 0) of whether each prediction was correct.
        num_bins (int): The number of bins to use for calibration.
    Returns:
        float: The Expected Calibration Error (ECE).
    
    """
    confidences = np.array(confidences)
    correctness = np.array(correctness)

    ece = 0.0
    bins = np.linspace(0, 1, num_bins + 1)

    for i in range(num_bins):
        lower, upper = bins[i], bins[i+1]
        idx = (confidences >= lower) & (confidences < upper)
        if np.sum(idx) == 0:
            continue
        bin_conf = np.mean(confidences[idx])
        bin_acc  = np.mean(correctness[idx])
        ece     += np.abs(bin_conf - bin_acc) * np.mean(idx)

    return ece

In [ ]:
def format_query(question, choices):
    """
    Formats the question and choices into a prompt for the model.
    Args:
        question (str): The question to be answered.
        choices (list of str): The list of answer choices.
    Returns:
        str: The formatted prompt for the model.
    """
    system_prompt = """You are a knowledgeable board-certified physician with strong a strong foundation of medical knowledge. You have taken several 
    medical exams to earn your certification. Respond ONLY with the letter corresponding to the correct answer.
    """
    text = system_prompt
    text += f"Question: {question}\n"
    
    for i, ch in enumerate(choices):
        letter = chr(ord('A') + i)
        text += f"{letter}. {ch}\n"
    
    text += "Answer:"
    return text

In [ ]:
def score_option(model, tokenizer, prompt, letter):
    """
    Scores a given answer option using the model.
    Args:
        model: The language model.
        tokenizer: The tokenizer for the model.
        prompt (str): The formatted prompt including the question and choices.
        letter (str): The answer letter to score (e.g., 'A', 'B', 'C', 'D').
    Returns:
        float: The score for the given answer option (higher is better)."""
    
    # Build answer text
    answer_text = f"Answer: {letter}"

    # Tokenize prompt and answer
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    answer_ids = tokenizer(answer_text, return_tensors="pt").input_ids.to(model.device)

    # Build full sequence: [prompt] + [answer]
    full_ids = torch.cat([prompt_ids, answer_ids], dim=1)

    # Create labels: ignore prompt tokens with -100
    labels = full_ids.clone()
    labels[:, :prompt_ids.shape[1]] = -100   # mask out prompt tokens

    # Compute loss (only on answer tokens)
    with torch.no_grad():
        out = model(input_ids=full_ids, labels=labels)
        nll = out.loss.item()

    return -nll   # higher = better


## Main Loop:
This loop will run the entire evaluation 

In [ ]:
correct_count = 0
confidences = []
correctness = []

print("Running evaluation...")

for i, item in enumerate(tqdm(processed, desc="Evaluating", ncols=80)):

    question = item["question"]
    options = item["options"]
    correct_answer_letter = item["answer"]
    correct_idx = ord(correct_answer_letter) - ord("A")

    # Build prompt WITH doctor persona
    prompt = format_query(question, options)
    
    letters = ["A", "B", "C", "D"]

    # Score each answer choice using full log-likelihood
    scores = [
        score_option(model, tokenizer, prompt, letter)
        for letter in letters
    ]

    # Choose highest scoring answer
    pred_idx = int(np.argmax(scores))
    conf = float(torch.softmax(torch.tensor(scores), dim=0)[pred_idx])
    is_correct = (pred_idx == correct_idx)

    pred_letter = letters[pred_idx]
    correct_letter = correct_answer_letter

    correct_count += is_correct
    confidences.append(conf)
    correctness.append(is_correct)


accuracy = correct_count / len(processed)
ece = expected_calibration_error(confidences, correctness)

print("----- RESULTS -----")
print(f"Accuracy: {accuracy:.4f}")
print(f"ECE:      {ece:.4f}")